# Контрольная работа №3. Диагностика моделей машинного обучения
Датасет `Volcanoes-a1`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, learning_curve, validation_curve
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_curve, auc, precision_recall_curve,
    classification_report
)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## Задание 1. Загрузка датасета

In [ ]:
dataset = fetch_openml(name='Volcanoes-a1', version=1, as_frame=True)
print(dataset.DESCR)

Датасет содержит числовые характеристики участков радарных снимков поверхности Венеры, полученных аппаратом Magellan.

Бизнес-задача: автоматически определять по радарному снимку, присутствует ли на данном участке вулкан. Ручная разметка тысяч снимков требует много времени эксперта-геолога.

Задача МО: бинарная классификация — по числовым признакам радарного изображения предсказать метку класса: 1 (вулкан есть) или 2 (вулкана нет).

In [ ]:
X = dataset.data
y = dataset.target
print('Признаки:', X.columns.tolist())
print('Размер датасета:', X.shape)

## Задание 2. Предварительный анализ

In [ ]:
df = X.copy()
df['Class'] = y
display(df.head())
display(df.describe().round(3))

In [ ]:
class_counts = y.value_counts()
print(pd.DataFrame({'Количество': class_counts, 'Доля': y.value_counts(normalize=True).round(3)}))

fig, ax = plt.subplots()
class_counts.plot(kind='bar', ax=ax, color=['#4C72B0', '#DD8452'])
ax.set_title('Распределение классов (1 — вулкан, 2 — не вулкан)')
ax.set_xlabel('Класс')
ax.set_ylabel('Количество')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

Выраженный дисбаланс классов: класс 2 (нет вулкана) составляет около 80% объектов, класс 1 (есть вулкан) — около 20%. При таком дисбалансе accuracy будет завышенной — модель может предсказывать всегда класс 2 и получать 80%. Важно смотреть на F1, Precision, Recall и PR-кривую.

## Задание 3. Проверка данных

In [ ]:
print('Пропущенных значений:')
print(df.isnull().sum())
print('\nТипы признаков:')
print(X.dtypes)

In [ ]:
le = LabelEncoder()
y_enc = le.fit_transform(y)
print('Классы и их коды:', dict(zip(le.classes_, range(len(le.classes_)))))

X_num = X.astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X_num, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')
print(f'Пропусков: {X_num.isnull().sum().sum()}')

Пропущенных значений нет, все признаки числовые — данные готовы к обучению. Разбивка стратифицирована, чтобы сохранить пропорцию классов в обеих частях.

## Задание 4. Линейная модель

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

print(f'score() = {lr.score(X_test, y_test):.4f}')

Метод `score()` для классификатора возвращает accuracy — долю правильно предсказанных объектов от общего числа. При дисбалансе классов accuracy может быть обманчивой: модель, предсказывающая всегда класс 2, получит ~80% без реального обучения. Поэтому смотрим дополнительные метрики.

## Задание 5. Ещё 3 метрики

In [ ]:
y_pred = lr.predict(X_test)

print(f'F1-score (weighted):  {f1_score(y_test, y_pred, average="weighted"):.4f}')
print(f'Precision (weighted): {precision_score(y_test, y_pred, average="weighted"):.4f}')
print(f'Recall (weighted):    {recall_score(y_test, y_pred, average="weighted"):.4f}')

print('\n', classification_report(y_test, y_pred, target_names=le.classes_))

Использую усреднение weighted, которое учитывает размер классов. F1 — гармоническое среднее precision и recall, хорошо работает при дисбалансе. Precision показывает, сколько из предсказанных вулканов реально вулканы. Recall — сколько реальных вулканов модель нашла. При обнаружении вулканов recall важнее: пропустить вулкан хуже, чем ложная тревога.

## Задание 6. ROC-кривая

In [ ]:
y_score = lr.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_score)
roc_auc_val = auc(fpr, tpr)

# оптимальный порог по критерию Юдена: max(TPR - FPR)
best_idx = np.argmax(tpr - fpr)
opt_thr  = thresholds[best_idx]

fig, ax = plt.subplots()
ax.plot(fpr, tpr, lw=2, color='steelblue', label=f'ROC-AUC = {roc_auc_val:.3f}')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Случайная модель')
ax.scatter(fpr[best_idx], tpr[best_idx], color='red', zorder=5,
           label=f'Оптимальный порог = {opt_thr:.3f}')
ax.set_title('ROC-кривая — Logistic Regression')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()
plt.tight_layout()
plt.show()

print(f'ROC-AUC:           {roc_auc_val:.4f}')
print(f'Оптимальный порог: {opt_thr:.4f}')
print(f'TPR при пороге:    {tpr[best_idx]:.4f}')
print(f'FPR при пороге:    {fpr[best_idx]:.4f}')

ROC-AUC показывает способность модели разделять классы. Оптимальный порог найден по критерию Юдена — точка с максимальным (TPR − FPR), обозначена красной точкой. По умолчанию логистическая регрессия использует порог 0.5, но при дисбалансе классов оптимальный порог часто смещается.

## Задание 7. PR-кривая

In [ ]:
precision_vals, recall_vals, pr_thresholds = precision_recall_curve(y_test, y_score)
pr_auc_val = auc(recall_vals, precision_vals)

# оптимальный порог: максимальный F1
f1_vals     = 2 * precision_vals[:-1] * recall_vals[:-1] / (precision_vals[:-1] + recall_vals[:-1] + 1e-9)
best_pr_idx = np.argmax(f1_vals)
opt_pr_thr  = pr_thresholds[best_pr_idx]

fig, ax = plt.subplots()
ax.plot(recall_vals, precision_vals, lw=2, color='steelblue', label=f'PR-AUC = {pr_auc_val:.3f}')
ax.scatter(recall_vals[best_pr_idx], precision_vals[best_pr_idx], color='red', zorder=5,
           label=f'Оптимальный порог = {opt_pr_thr:.3f}')
ax.set_title('PR-кривая — Logistic Regression')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend()
plt.tight_layout()
plt.show()

print(f'PR-AUC:            {pr_auc_val:.4f}')
print(f'Оптимальный порог: {opt_pr_thr:.4f}')

PR-кривая при дисбалансе информативнее ROC: она не учитывает истинно-отрицательные объекты и лучше отражает качество предсказания minority-класса (вулкан). Оптимальный порог найден как точка максимального F1.

## Задание 8. Кросс-валидация

Для бинарной классификации с дисбалансом выбрал Stratified K-Fold: он сохраняет пропорцию классов в каждом фолде. Без стратификации при соотношении 80/20 возможны фолды, где класс 1 представлен очень мало, что даёт некорректную оценку.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(rf, X_num, y_enc, cv=cv, scoring='accuracy')

print('Random Forest, Stratified 5-Fold CV:')
print(f'Accuracy по фолдам: {[round(s, 4) for s in cv_scores]}')
print(f'Среднее: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

Random Forest показывает стабильное качество по всем фолдам, разброс небольшой. Качество выше логистической регрессии, так как границы классов нелинейные.

## Задание 9. Кривые обучения

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    rf, X_num, y_enc,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy',
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots()
ax.plot(train_sizes, train_mean, 'o-', color='steelblue', label='Обучающая')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2, color='steelblue')
ax.plot(train_sizes, val_mean, 'o-', color='tomato', label='Валидационная')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.2, color='tomato')
ax.set_title('Кривые обучения — Random Forest')
ax.set_xlabel('Размер обучающей выборки')
ax.set_ylabel('Accuracy')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Train accuracy: {train_mean[-1]:.4f}')
print(f'Val accuracy:   {val_mean[-1]:.4f}')
print(f'Разрыв:         {train_mean[-1] - val_mean[-1]:.4f}')

Обучающая кривая держится около 1.0, валидационная ниже — есть умеренное переобучение. С ростом числа объектов валидационная кривая растёт, то есть больше данных помогает. Признаков недообучения нет.

## Задание 10. Влияние гиперпараметра

In [ ]:
param_range = [1, 2, 3, 5, 7, 10, 15, 20]

train_scores_v, val_scores_v = validation_curve(
    DecisionTreeClassifier(random_state=42),
    X_num, y_enc,
    param_name='max_depth',
    param_range=param_range,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy',
    n_jobs=-1
)

train_mean_v = train_scores_v.mean(axis=1)
val_mean_v   = val_scores_v.mean(axis=1)
train_std_v  = train_scores_v.std(axis=1)
val_std_v    = val_scores_v.std(axis=1)

fig, ax = plt.subplots()
ax.plot(param_range, train_mean_v, 'o-', color='steelblue', label='Обучающая')
ax.fill_between(param_range, train_mean_v - train_std_v, train_mean_v + train_std_v, alpha=0.2, color='steelblue')
ax.plot(param_range, val_mean_v, 'o-', color='tomato', label='Валидационная')
ax.fill_between(param_range, val_mean_v - val_std_v, val_mean_v + val_std_v, alpha=0.2, color='tomato')
ax.set_title('Влияние max_depth — Decision Tree')
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_xticks(param_range)
ax.legend()
plt.tight_layout()
plt.show()

best_depth = param_range[np.argmax(val_mean_v)]
print(f'Лучший max_depth по валидации: {best_depth}, accuracy = {max(val_mean_v):.4f}')

print(pd.DataFrame({
    'max_depth': param_range,
    'train':     train_mean_v.round(4),
    'val':       val_mean_v.round(4),
    'gap':       (train_mean_v - val_mean_v).round(4)
}).to_string(index=False))

При малых значениях max_depth дерево недообучается. При увеличении глубины валидационная точность достигает максимума, после чего начинает снижаться — дерево переобучается, запоминает тренировочные примеры. При дисбалансе классов переобучение особенно заметно, так как модель начинает запоминать редкий класс вместо обобщения.